# 00 · Spark basics for SQL users

**Start here** if you know SQL (Hive, Impala) but have never used Spark or Python notebooks.

**Goal:** by the end you can read every code cell in notebooks 01 and 02, and write simple Spark queries yourself — in SQL *and* in Python.

**Time:** about 45 minutes.

### How a notebook works (2-minute primer)
- The page is a list of **cells**. Grey boxes are **code cells**; text like this is a **markdown cell**.
- Click a code cell and press **Shift + Enter** to run it. Its output appears right below, and the cursor jumps to the next cell.
- Run cells **in order, top to bottom**. A later cell often uses something an earlier cell created (like a table name or a variable).
- `[*]` next to a cell means it's still running. `[5]` means it was the 5th cell run.
- If things get confusing: menu **Kernel → Restart Kernel and Clear Outputs**, then run from the top again.

## 1. What Spark is

**Spark is a distributed SQL and data-processing engine.** You give it a query; it splits the work into many small **tasks**, runs them in parallel, and combines the results.

| | Hive (on MapReduce/Tez) | Impala | Spark |
|---|---|---|---|
| What it is | SQL → batch jobs | SQL engine with long-running daemons | SQL **and** a programming API (Python, Scala, Java) |
| Language | SQL only | SQL only | SQL **or** Python/Scala code — mix freely |
| Where it runs | YARN | Impala daemons | YARN, Kubernetes, Databricks — or **one machine** |

Spark has two kinds of processes:
- **Driver** — the "coordinator". Turns your query into a plan and hands out tasks. (Like Impala's coordinator node.)
- **Executors** — the "workers" that run tasks on chunks of data. (Like Impala daemons doing scans.)

**In this lab, Spark runs in *local mode*:** the driver and executors live in one process inside the `spark` container, and "executors" are just threads — one per CPU. Same code you'd run on a 100-node Databricks cluster; it just has fewer workers.

## 2. The SparkSession — your connection to Spark

Every Spark program starts by creating a **SparkSession**, usually stored in a variable called `spark`. Think of it as your open **beeline / impala-shell connection**: everything you do goes through it.

**What this cell does**
- `from lab_helpers import get_spark` — loads a small helper function from the file `lab_helpers.py` (next to this notebook). `import` is how Python loads code from another file or library.
- `spark = get_spark("00-spark-basics")` — starts Spark and stores the session in a variable named `spark`. The text in quotes is just a name that shows up in the Spark UI.
- `print(...)` — displays values. `spark.version` reads the Spark version from the session.

In [ ]:
from lab_helpers import get_spark

spark = get_spark("00-spark-basics")
print("Spark version:", spark.version)
print("Running on:   ", spark.sparkContext.master)
print("Parallel slots:", spark.sparkContext.defaultParallelism)

**What to notice**
- `local[*]` = local mode, using all CPUs given to the container (`*`).
- **Parallel slots** = how many tasks can run at the same time (one per CPU). On a real cluster this would be hundreds.
- The line **Spark UI for this notebook: http://localhost:4040** is printed by `get_spark()`. Open that link in another tab; we'll use it in section 7.
- Ignore `WARNING: Using incubator modules` — it's Java announcing an optional feature, not a problem.
- If you see `Service 'SparkUI' could not bind on port 4040. Attempting port 4041`, another open notebook already has a Spark session using port 4040. That's fine: this notebook's UI is on 4041, and the link printed above already points to it.

> `get_spark()` is 3 lines long. It calls Spark's own `SparkSession.builder.appName(...).getOrCreate()` and hides Spark's log chatter. All the settings (where the catalog and storage are) come from `docker/spark/conf/spark-defaults.conf`, so you never type them.

## 3. Your first query — in SQL

`spark.sql("...")` runs **any SQL** and gives back a result. `.show()` prints the first rows.

**What this cell does**
- Spark SQL can query a file **directly** by putting the format and path in back-ticks: `` parquet.`/path/file.parquet` ``. No `CREATE TABLE` needed.
- The SQL is inside `"""` triple quotes `"""`, which lets a Python string span several lines.

In [ ]:
spark.sql("""
    SELECT VendorID, tpep_pickup_datetime, trip_distance, fare_amount
    FROM parquet.`/home/lab/work/data/raw/yellow_tripdata_2024-01.parquet`
    LIMIT 5
""").show()

**What to notice:** ordinary SQL, ordinary table output. If you only ever wrote SQL inside `spark.sql(...)`, you could do most of this lab.

## 4. The same data as a DataFrame

A **DataFrame** is Spark's name for "a table-shaped result": rows and named, typed columns. Every `spark.sql(...)` returns one, and you can also create one by reading files.

**What this cell does**
- `spark.read.parquet(path)` — "read this Parquet file as a DataFrame". Similar to `CREATE EXTERNAL TABLE ... LOCATION` — but nothing is registered anywhere; it's just a variable in Python.
- `trips = ...` — keeps the DataFrame in a variable named `trips` so later cells can use it.
- `trips.printSchema()` — like `DESCRIBE trips`.

In [ ]:
RAW = "/home/lab/work/data/raw/yellow_tripdata_2024-01.parquet"   # a variable holding the file path

trips = spark.read.parquet(RAW)
trips.printSchema()

**What to notice:** Spark read the column names and types from the Parquet file itself. (`long` = `BIGINT`, `integer` = `INT`, `timestamp_ntz` = timestamp without time zone.)

## 5. Lazy evaluation — the most important Spark idea

Spark splits operations into two kinds:

| Kind | Examples | What happens |
|---|---|---|
| **Transformation** | `select`, `filter`, `groupBy`, `join`, `orderBy` | Nothing runs. Spark only **adds a step to the plan** and returns a new DataFrame instantly. |
| **Action** | `show`, `count`, `collect`, `write`, `toPandas` | Spark **optimizes the whole plan and runs it**. |

It's like writing a `CREATE VIEW`: defining the view is instant; the work happens when someone queries it. Waiting until an action lets Spark optimize the **whole** chain at once (e.g. read only needed columns, push filters down to the file).

**What this cell does**
- `%%time` (first line) is a Jupyter feature that times the cell.
- `trips.filter("fare_amount > 100")` — a transformation: keep rows where the condition is true. The condition is written exactly like a SQL `WHERE` clause.

In [ ]:
%%time
expensive = trips.filter("fare_amount > 100")     # transformation: instant, nothing read yet
print(type(expensive))

In [ ]:
%%time
expensive.count()                                 # action: now Spark actually reads the file

**What to notice:** the first cell took **milliseconds** — Spark didn't touch the data. The `count()` took noticeably longer: that's when the file was read and filtered. If a cell "does nothing" and returns instantly, it probably only built a plan.

## 6. SQL and the DataFrame API side by side

Everything you can write in SQL, you can write with DataFrame methods — and vice versa. Both produce **the same plan** and run at the same speed. Use whichever reads better.

To use SQL on a DataFrame, give it a name with **`createOrReplaceTempView`**. A temp view exists only in this session — like a `CREATE TEMPORARY VIEW` in Hive.

**What this cell does**
- Registers the `trips` DataFrame under the SQL name `trips`.
- Imports `functions` from PySpark under the short name `F`. You'll see `F.something` everywhere — `F.sum`, `F.avg`, `F.col` — they are SQL functions for the DataFrame API.

In [ ]:
from pyspark.sql import functions as F

trips.createOrReplaceTempView("trips")

### 6a. Filter and pick columns

| SQL | DataFrame API |
|---|---|
| `SELECT a, b FROM t WHERE x > 10` | `t.select("a", "b").filter("x > 10")` |

Run both cells and compare.

In [ ]:
spark.sql("""
    SELECT PULocationID, fare_amount
    FROM trips
    WHERE fare_amount > 500
""").show(3)

In [ ]:
trips.select("PULocationID", "fare_amount").filter("fare_amount > 500").show(3)

**Reading the Python:** `trips.select(...).filter(...).show(3)` is **method chaining** — each method returns a new DataFrame, and the next method is called on it. Read it left to right like SQL clauses. `show(3)` prints 3 rows.

### 6b. Group, aggregate, sort

**Reading the Python below**
- The whole expression is wrapped in `( ... )` so it can be split over several lines. Python ignores line breaks inside brackets.
- `F.count("*").alias("trips")` = `count(*) AS trips`.
- `F.round(F.avg("fare_amount"), 2)` = `round(avg(fare_amount), 2)`.
- `F.desc("trips")` = `ORDER BY trips DESC`.

In [ ]:
spark.sql("""
    SELECT payment_type,
           count(*)                   AS trips,
           round(avg(fare_amount), 2) AS avg_fare
    FROM trips
    GROUP BY payment_type
    ORDER BY trips DESC
""").show()

In [ ]:
(trips
    .groupBy("payment_type")
    .agg(F.count("*").alias("trips"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare"))
    .orderBy(F.desc("trips"))
    .show())

**What to notice:** identical results. Payment types are codes: 1 = credit card, 2 = cash, 3 = no charge, 4 = dispute; 0 means it was not recorded.

### 6c. Join with a lookup table

The zone lookup is a CSV file: `LocationID → Borough, Zone`.

**Reading the Python below**
- `spark.read.option("header", True).option("inferSchema", True).csv(path)` — read a CSV; the first line has column names; let Spark guess the types. (Parquet never needs these options — it stores its own schema.)
- `trips.join(zones, trips.PULocationID == zones.LocationID)` = `FROM trips JOIN zones ON trips.PULocationID = zones.LocationID`. `==` means "equals" in Python (a single `=` assigns a variable).

In [ ]:
zones = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv("/home/lab/work/data/raw/taxi_zone_lookup.csv"))
zones.createOrReplaceTempView("zones")
zones.show(3)

In [ ]:
spark.sql("""
    SELECT z.Borough, count(*) AS pickups
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY pickups DESC
""").show()

In [ ]:
(trips.join(zones, trips.PULocationID == zones.LocationID)
      .groupBy("Borough")
      .agg(F.count("*").alias("pickups"))
      .orderBy(F.desc("pickups"))
      .show())

**What to notice:** same answer again. Manhattan dominates pickups. `Unknown` (zone 264) is used when the meter couldn't record a location, and `N/A` (zone 265, "Outside of NYC") for pickups outside the city. Real data always has values like these. In the rest of the lab you'll see both styles; whenever a Python line confuses you, ask "which SQL clause is this?"

## 7. How Spark runs a query: partitions, tasks, the plan

Spark splits data into **partitions** — chunks processed independently, one **task** per partition. (Like HDFS blocks → mapper tasks, or Impala scan ranges.)

**What this cell does**
- `trips.rdd.getNumPartitions()` — how many chunks Spark split the file into.
- `.explain()` — prints the **physical plan**, like `EXPLAIN` in Impala. Read it **bottom to top**: the scan is at the bottom, the final step at the top.

In [ ]:
print("partitions:", trips.rdd.getNumPartitions())

by_borough = (trips.join(zones, trips.PULocationID == zones.LocationID)
                   .groupBy("Borough").count())
by_borough.explain()

**What to notice** (bottom to top)
- `FileScan parquet` and `FileScan csv` — reading trips and the zones file. `ReadSchema: struct<PULocationID:int>`: from the 19-column trips file Spark reads **only** the one column this query needs.
- `BroadcastHashJoin` / `BroadcastExchange` — the zones table is tiny, so Spark **sends a copy of it to every task** instead of shuffling the big table. Same idea as Impala's broadcast join.
- `HashAggregate` appears **twice**, with an `Exchange hashpartitioning(Borough...)` between them: each task pre-aggregates its chunk, then rows are **shuffled** by `Borough` so each borough ends up in one place for the final count. A shuffle = data moving between tasks; it's usually the most expensive step. The `8` in `hashpartitioning(Borough, 8)` is the number of shuffle partitions, set to 8 for a laptop in `spark-defaults.conf` (Spark's default of 200 is meant for clusters).
- `AdaptiveSparkPlan isFinalPlan=false` (top line) — Spark may **re-plan while running**, using real sizes it learns along the way (Adaptive Query Execution). This plan is the starting point.

Now run the action and look at the Spark UI: http://localhost:4040 → **Jobs** → click the latest job → you'll see **stages** split at the shuffle, and the number of tasks per stage.

In [ ]:
by_borough.show()

## 8. Getting results into Python: `show` vs `collect` vs `toPandas`

| Method | What you get | Safe for big data? |
|---|---|---|
| `df.show(n)` | Prints `n` rows (default 20) | ✔ only fetches `n` rows |
| `df.count()` | One number | ✔ |
| `df.collect()` | **All rows** as a Python list on the driver | ✘ only for small results |
| `df.toPandas()` | **All rows** as a pandas table (nice display in notebooks) | ✘ only for small results |
| `df.first()` | One row | ✔ |

`collect()` / `toPandas()` pull every row into the driver's memory — like fetching millions of rows into `impala-shell`. Fine for an aggregate with 10 rows; a crash for 3 million.

**What this cell does**
- `by_borough.toPandas()` — the result has ~7 rows, so it's safe. In a notebook, a pandas table displays as a formatted table.
- `trips.agg(...).first()[0]` — one row back; `[0]` takes its first value.

In [ ]:
max_fare = trips.agg(F.max("fare_amount")).first()[0]
print("Highest fare in January:", max_fare)

by_borough.orderBy(F.desc("count")).toPandas()

## 9. Catalogs, namespaces and tables

So far we queried **files**. Real pipelines use **tables** registered in a catalog — that's where Iceberg comes in (notebook 02).

| Spark term | Hive term | In this lab |
|---|---|---|
| **catalog** | (the metastore) | `lake` = the Iceberg REST catalog service |
| **namespace** | database | e.g. `learn`, later `bronze`, `silver`, `gold` |
| **table** | table | e.g. `learn.yellow_trips` |

A full table name is `catalog.namespace.table` — `lake.learn.yellow_trips`. Because `lake` is set as the default catalog, `learn.yellow_trips` is enough. Temp views (like `trips` above) are *not* in any catalog; they vanish when the session ends.

In [ ]:
print("default catalog:", spark.catalog.currentCatalog())
spark.sql("SHOW NAMESPACES").show()

**What to notice:** the default catalog is `lake`. The namespace list is empty on a fresh lab, or shows `learn` if you've already run notebook 02. `SHOW NAMESPACES` is the same as Hive's `SHOW DATABASES`.

## 10. Python you'll meet in this lab — cheat sheet

| You see | It means |
|---|---|
| `x = 5` | Store a value in a variable named `x` |
| `"text"` or `'text'` | A string. `"""multi-line text"""` spans lines |
| `f"rows: {n}"` | An **f-string**: `{n}` is replaced by the value of `n` |
| `# comment` | Ignored by Python |
| `print(a, b)` | Display values |
| `df.method(...)` | Call a method on an object (e.g. a DataFrame) |
| `( ... )` around many lines | Lets one statement span lines |
| `[a, b, c]` | A **list**; `mylist[0]` is the first item |
| `{"key": value}` | A **dict** (key → value lookup) |
| `for x in items:` + indented lines | Repeat the indented lines for each item |
| `def name(args):` + indented lines | Define a function |
| `import x` / `from x import y` | Load a library |

## Summary

| Idea | One line |
|---|---|
| SparkSession (`spark`) | Your connection to Spark |
| DataFrame | A table-shaped result; built lazily |
| Transformation vs action | Transformations plan; actions run |
| `spark.sql(...)` vs DataFrame API | Same engine, same plan — pick what reads better |
| Temp view | Give a DataFrame a SQL name for this session |
| Partition / task / shuffle | Chunk of data / unit of work / data moving between tasks |
| `explain()` + Spark UI | See how a query runs |
| `show` vs `collect` | Peek safely vs pull everything to the driver |

## Exercises
1. Find the 5 pickup zones (`Zone`) with the highest average `tip_amount`, using SQL in the cell below. Then write the same query with the **DataFrame API**.
   This looks like a simple `GROUP BY`, but **which trips you include changes the answer**. Decide, and explain your choice in a `--` comment:
   - **Payment type:** tips are only recorded for credit-card trips (`payment_type = 1`); cash tips appear as 0. Should cash trips count?
   - **Small zones:** a zone with 2 trips and one big tip would top the list. Require a minimum number of trips (`HAVING count(*) >= ...`)?
   - **Bad rows:** you've seen negative fares. Are there negative tips to exclude?
2. Run `trips.filter("trip_distance > 50")` on its own. Did Spark read any data? How can you tell? Now add `.count()`.
3. Run `.explain()` on your exercise-1 query. Find the join type and the `Exchange` (shuffle) steps.

Next: **`01_parquet_basics.ipynb`**.

In [ ]:
# Exercise 1 — the 5 pickup zones with the highest average tip.
# The temp views `trips` and `zones` from sections 6 and 6c are available here.
top_tip_zones = spark.sql("""
    -- TODO(human): write the query. Decide which trips should count (see the hints above).
    SELECT 'replace me' AS Zone
""")
top_tip_zones.show()